In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
import os
import random
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from torchvision.transforms import Compose, ToTensor, Normalize, Resize

In [7]:
import os
os.chdir('/media/sslab/PACS/sslab/nguyentiendung/data')

In [8]:
df = pd.read_excel("participants.xlsx")
df_test = df[(df["No."] >= 1500) & (df["No."] <= 2000)].reset_index(drop=True)
df_test

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1500,21.0,control,m,sub-BrainAge019025,INDI/SALD
1,1501,21.0,control,m,sub-BrainAge019026,INDI/SALD
2,1502,49.0,control,f,sub-BrainAge019027,INDI/SALD
3,1503,37.0,control,m,sub-BrainAge019028,INDI/SALD
4,1504,20.0,control,m,sub-BrainAge019029,INDI/SALD
...,...,...,...,...,...,...
496,1996,21.0,control,f,sub-BrainAge019533,INDI/SLIM
497,1997,22.0,control,f,sub-BrainAge019534,INDI/SLIM
498,1998,20.0,control,f,sub-BrainAge019535,INDI/SLIM
499,1999,20.0,control,f,sub-BrainAge019536,INDI/SLIM


In [19]:
class BrainAgeDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
        self.target_size = (128, 128)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        img_3d = nib.load(sample["image_path"]).get_fdata()

        img_3d = (img_3d - img_3d.min()) / (img_3d.max() - img_3d.min() + 1e-8)

        d, h, w = img_3d.shape
        axial = resize(img_3d[d // 2, :, :], self.target_size, mode='reflect', anti_aliasing=True)
        coronal = resize(img_3d[:, h // 2, :], self.target_size, mode='reflect', anti_aliasing=True)
        sagittal = resize(img_3d[:, :, w // 2], self.target_size, mode='reflect', anti_aliasing=True)

        img_rgb = np.stack([axial, coronal, sagittal], axis=-1)
        img_rgb = (img_rgb * 255).astype(np.uint8)

        img_pil = Image.fromarray(img_rgb)

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = ToTensor()(img_pil)

        return {
            "image": img_tensor,
            "age": torch.tensor(sample["subject_age"], dtype=torch.float32),
            "subject_id": sample["subject_id"]
        }

In [21]:
test_hf_dataset = []
for _, row in df_test.iterrows():
    subj_id = row["subject_id"]
    age = row["subject_age"]
    nii_path = f"data/{subj_id}/anat/{subj_id}_T1w.nii.gz"
    test_hf_dataset.append({"image_path": nii_path, "subject_age": age, "subject_id": subj_id})

test_hf_dataset[0:5]

[{'image_path': 'data/sub-BrainAge019025/anat/sub-BrainAge019025_T1w.nii.gz',
  'subject_age': 21.0,
  'subject_id': 'sub-BrainAge019025'},
 {'image_path': 'data/sub-BrainAge019026/anat/sub-BrainAge019026_T1w.nii.gz',
  'subject_age': 21.0,
  'subject_id': 'sub-BrainAge019026'},
 {'image_path': 'data/sub-BrainAge019027/anat/sub-BrainAge019027_T1w.nii.gz',
  'subject_age': 49.0,
  'subject_id': 'sub-BrainAge019027'},
 {'image_path': 'data/sub-BrainAge019028/anat/sub-BrainAge019028_T1w.nii.gz',
  'subject_age': 37.0,
  'subject_id': 'sub-BrainAge019028'},
 {'image_path': 'data/sub-BrainAge019029/anat/sub-BrainAge019029_T1w.nii.gz',
  'subject_age': 20.0,
  'subject_id': 'sub-BrainAge019029'}]

In [16]:
transform = Compose([
    Resize((128, 128)),
    ToTensor(),
    Normalize(mean=[0.5]*3, std=[0.5]*3)
])

In [18]:
test_dataset = BrainAgeDataset(test_hf_dataset, transform=transform)
test_dataset[0]

FileNotFoundError: No such file or no access: 'data/sub-BrainAge019025/anat/sub-BrainAge019025_T1w.nii.gz'

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2)
        )
        self.regressor = nn.Sequential(
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.regressor(x)
        return x.squeeze()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
model.load_state_dict(torch.load("model.pt", map_location=device))
model.eval()